# Paper 2 — Stage 4: Construct Validity (Matched-Marker Probe)

**Runtime: CPU, hours. N_BOOT = 2000. Resumable via checkpoints.**

**This notebook produces Table 1 — the headline construct-validity result.** Run it before any downstream analysis: it is the check everything else rests on.

Three contrasts per model:

| Arm | What it compares | Why |
|---|---|---|
| ORIGINAL | revision windows vs. arbitrary control windows | confounded — marker word differs |
| MATCHED | revision-`wait` vs. filler-`wait` | the real test — marker held constant |
| STRICT NULL | filler vs. filler | must land at chance, or the pipeline leaks |

The reported quantity is the **residual** (MATCHED − STRICT NULL) with bootstrap CIs.

Uses `StratifiedGroupKFold` with original-event-ID groups. This matters: an earlier version split folds *after* bootstrap resampling, which let ~45% of test-fold rows see their own duplicate in training and inflated accuracy by ~+0.037.

**Prerequisites:** `activations/` (Stage 3) and `activations_filler/` (notebook 1).

> **Which version this is.** Built from `task1_v4_corrected.py`, which adds within-draw residual pairing — the fix for the Qwen-7B point-estimate/CI inconsistency. `task1_v3_corrected.py` is the version whose real-data run was actually observed and which produced the originally-reported table; v4 was written and validated on synthetic data but its real run was not observed. If your numbers need to match a specific earlier table, check which one you are reproducing.


In [ ]:
# ---- Google Drive -----------------------------------------------------------
from google.colab import drive
drive.mount("/content/gdrive")

## Probe


In [ ]:
# =============================================================================
# TASK 1 (v3) -- headline result at N_BOOT = 2000, corrected per pressure-test
#               review: eliminates duplicate-event train/test leakage inside
#               bootstrap replicates, averages the point estimate over
#               multiple balanced draws, and records the per-layer accuracy
#               profile for the Qwen-14B peak-layer check.
#
# CHANGES FROM v2:
#   1. [Task A -- CORRECTNESS, was silently biasing the CIs] cv_accuracy now
#      takes an original-event-id ("group") array through every subsampling
#      step and splits with StratifiedGroupKFold instead of StratifiedKFold.
#      Verified on synthetic data: the old split let ~45% of test-fold rows
#      have their exact duplicate sitting in the training fold whenever the
#      bootstrap had resampled-with-replacement data, which inflated
#      bootstrap-mean accuracy by ~+0.03-0.04 and pushed the point estimate
#      out to roughly the 20th percentile of its own bootstrap distribution.
#      With the group-aware split, overlap is exactly 0 and the point
#      estimate sits much closer to the center of its distribution.
#   2. [Task B] Point estimate for matched/null is now the mean of 30
#      independent balanced draws (not a single draw), removing the
#      single-draw variance the v2 point estimate carried.
#   3. [Task C / Fix 4] peak_layer now returns and saves the full per-layer
#      accuracy profile, plus a flatness flag, so the Qwen-14B layer-47
#      argmax can be checked against its neighbors rather than trusted blind.
#
# RUNTIME: similar order to v2; Task B adds ~30x calls to cv_accuracy for the
#          point estimate only (bootstrap replicates unchanged at 1x each).
# =============================================================================



from pathlib import Path
GDRIVE_BASE = Path("/content/gdrive/MyDrive/llm_planning_study")
assert (GDRIVE_BASE / "activations").exists(),        "activations/ missing -- bad mount"
assert (GDRIVE_BASE / "activations_filler").exists(), "fillers missing -- bad mount"
print("Drive mounted, data visible.")

import json, gc
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold
import warnings
warnings.filterwarnings("ignore")

ACTIV_BASE  = GDRIVE_BASE / "activations"
FILLER_BASE = GDRIVE_BASE / "activations_filler"
RESULTS_DIR = GDRIVE_BASE / "results_v3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
MODELS = ["DeepSeek-R1-Distill-Llama-8B", "DeepSeek-R1-Distill-Qwen-1.5B",
          "DeepSeek-R1-Distill-Qwen-7B",  "DeepSeek-R1-Distill-Qwen-14B"]

CENTER       = 30
SEED         = 42
N_PCA        = 50          # pre-committed headline dimensionality
MAX_ITER     = 2000
N_BOOT       = 2000
N_PT_DRAWS   = 30          # Task B: averaged point-estimate draws
FLAT_WINDOW  = 0.02        # Task C: layers within this of peak are "flat with it"

OUTFILE = RESULTS_DIR / "task1_residual_cis_nboot2000_v3.json"

In [ ]:
def load_rev(m):
    rev = []
    for p in sorted((ACTIV_BASE / m).glob("*.npz")):
        rw = np.load(p).get("revision_windows")
        if rw is not None:
            rev.extend(rw[i] for i in range(rw.shape[0]))
    return rev

def load_fil(m):
    fil = []
    for p in sorted((FILLER_BASE / m).glob("*.npz")):
        w = np.load(p).get("filler_windows")
        if w is not None:
            fil.extend(w[i] for i in range(w.shape[0]))
    return fil


# -----------------------------------------------------------------------
# Task A: cv_accuracy now threads original-event-id groups through every
# subsampling step and uses StratifiedGroupKFold. Every array of features
# passed in (Xp, Xn) must be accompanied by a same-length array of original
# event IDs (gp, gn) so duplicates introduced by bootstrap resampling never
# get split across train/test.
# -----------------------------------------------------------------------
def cv_accuracy(Xp, Xn, gp, gn, seed):
    """Balanced 5-fold, group-aware CV accuracy. PCA fit inside each fold."""
    n = min(len(Xp), len(Xn))
    if n < 20:
        return 0.5
    rng = np.random.RandomState(seed)
    idx_p = rng.choice(len(Xp), n, False)
    idx_n = rng.choice(len(Xn), n, False)
    X = np.vstack([Xp[idx_p], Xn[idx_n]]).astype(np.float32)
    X = np.nan_to_num(X, nan=0., posinf=0., neginf=0.)
    y = np.array([1]*n + [0]*n)
    g = np.concatenate([gp[idx_p], gn[idx_n]])

    ncomp = min(N_PCA, len(y) - 1)
    pipe = Pipeline([
        ("sc",  StandardScaler()),
        ("pca", PCA(n_components=ncomp, random_state=seed)),
        ("clf", LogisticRegression(max_iter=MAX_ITER, random_state=seed)),
    ])
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=seed)
    accs = []
    for tr, te in cv.split(X, y, groups=g):
        pipe.fit(X[tr], y[tr])
        accs.append((pipe.predict(X[te]) == y[te]).mean())
    return float(np.mean(accs))


def averaged_point_estimate(Xp, Xn, gp, gn, base_seed, n_draws=N_PT_DRAWS):
    """Task B: point estimate = mean cv_accuracy over n_draws independent
    balanced subsample draws, so it isn't hostage to one lucky split."""
    vals = [cv_accuracy(Xp, Xn, gp, gn, base_seed + 10_000 + d) for d in range(n_draws)]
    return float(np.mean(vals)), vals


def paired_residual_point_estimate(Xr, Xf, gid_r, gid_f, base_seed, n_draws=N_PT_DRAWS):
    """
    Residual-pairing fix. The bootstrap CI for the residual is built from
    WITHIN-REPLICATE differences (matched and null computed on the same
    resampled filler pool each replicate), which inherits their correlation
    and is narrower than an independent-sum construction would suggest.
    The old residual point estimate (matched_pt - null_pt, both averaged
    separately over their own draws) did not share that pairing and could
    land outside its own CI (confirmed on real data: Qwen-7B).

    Fix: for each of n_draws, draw ONE balanced filler subsample and use it
    for BOTH the matched negative class and the null split, exactly mirroring
    what each bootstrap replicate does. Difference the two within that draw,
    then average the per-draw differences. This is the residual's own point
    estimate; it does not have to (and generally won't) equal the difference
    of the separately-reported matched_pt and null_pt marginals.
    """
    resids = []
    for d in range(n_draws):
        seed = base_seed + 10_000 + d
        rng = np.random.RandomState(seed)
        n = min(len(Xr), len(Xf))
        idx_r = rng.choice(len(Xr), n, False)
        idx_f = rng.choice(len(Xf), n, False)
        Xr_d, gr_d = Xr[idx_r], gid_r[idx_r]
        Xf_d, gf_d = Xf[idx_f], gid_f[idx_f]   # shared draw, used for both terms below

        m = cv_accuracy(Xr_d, Xf_d, gr_d, gf_d, seed)

        pf = np.random.RandomState(seed).permutation(len(Xf_d))
        h = len(Xf_d) // 2
        n_ = cv_accuracy(Xf_d[pf[:h]], Xf_d[pf[h:2*h]], gf_d[pf[:h]], gf_d[pf[h:2*h]], seed)

        resids.append(m - n_)
    return float(np.mean(resids)), resids


def peak_layer(rev, fil, nl):
    """Task C: also returns the full per-layer accuracy profile."""
    profile = []
    best_l, best_a = 0, 0.0
    rev_ids = np.arange(len(rev))
    fil_ids = np.arange(len(fil)) + 10**7   # disjoint id space from revisions
    for l in range(nl):
        Xp = np.array([w[l, CENTER] for w in rev], np.float32)
        Xn = np.array([w[l, CENTER] for w in fil], np.float32)
        a = cv_accuracy(Xp, Xn, rev_ids, fil_ids, SEED)
        profile.append(a)
        if a > best_a:
            best_a, best_l = a, l
    near_peak = [l for l, acc in enumerate(profile) if best_a - acc < FLAT_WINDOW]
    is_flat = len(near_peak) > 3
    return best_l, best_a, profile, near_peak, is_flat


def bootstrap_all(rev, fil, L):
    """
    Returns point estimates (Task B: averaged) and percentile CIs for
    matched, null, and residual. Bootstrap resamples EVENTS with replacement
    and REFITS both probes each iteration; original event IDs are carried
    through the resample step so cv_accuracy's group-aware split (Task A)
    can prevent the same event landing on both sides of a fold.
    """
    Xr = np.array([w[L, CENTER] for w in rev], dtype=np.float32)
    Xf = np.array([w[L, CENTER] for w in fil], dtype=np.float32)
    gid_r = np.arange(len(Xr))
    gid_f = np.arange(len(Xf)) + 10**7

    # ---- point estimates (Task B: averaged over N_PT_DRAWS balanced draws) ----
    matched_pt, matched_draws = averaged_point_estimate(Xr, Xf, gid_r, gid_f, SEED)

    permf = np.random.RandomState(SEED).permutation(len(Xf))
    hf    = len(Xf) // 2
    Xf_a, Xf_b_half = Xf[permf[:hf]], Xf[permf[hf:2*hf]]
    gf_a, gf_b_half = gid_f[permf[:hf]], gid_f[permf[hf:2*hf]]
    # null halves are disjoint filler subsets -> give them disjoint synthetic
    # group ids too, since they're different physical events by construction
    null_pt, null_draws = averaged_point_estimate(Xf_a, Xf_b_half, gf_a, gf_b_half, SEED)

    # Residual point estimate computed via within-draw pairing (not matched_pt - null_pt),
    # so it shares the bootstrap's correlation structure. matched_pt/null_pt above are
    # still reported as-is for the matched/null columns and their own CI checks.
    resid_pt, resid_pt_draws = paired_residual_point_estimate(Xr, Xf, gid_r, gid_f, SEED)

    # ---- bootstrap ----
    rng = np.random.RandomState(SEED)
    matched_b, null_b, resid_b = [], [], []
    for b in range(N_BOOT):
        ir  = rng.choice(len(Xr), len(Xr), replace=True)
        iff = rng.choice(len(Xf), len(Xf), replace=True)
        Xr_b, gr_b = Xr[ir], gid_r[ir]
        Xf_b, gf_b = Xf[iff], gid_f[iff]

        m_b = cv_accuracy(Xr_b, Xf_b, gr_b, gf_b, SEED + b)              # refit, group-aware

        pf  = np.random.RandomState(SEED + b).permutation(len(Xf_b))
        h   = len(Xf_b) // 2
        n_b = cv_accuracy(Xf_b[pf[:h]], Xf_b[pf[h:2*h]],
                           gf_b[pf[:h]], gf_b[pf[h:2*h]], SEED + b)      # refit, group-aware

        matched_b.append(m_b)
        null_b.append(n_b)
        resid_b.append(m_b - n_b)

        if (b + 1) % 250 == 0:
            print(f"      boot {b+1}/{N_BOOT}")

    def ci(arr):
        lo, hi = np.percentile(arr, [2.5, 97.5])
        return float(lo), float(hi)

    m_lo, m_hi = ci(matched_b)
    n_lo, n_hi = ci(null_b)
    r_lo, r_hi = ci(resid_b)

    return {
        "matched":      matched_pt, "matched_ci":  [m_lo, m_hi],
        "null_fil":     null_pt,    "null_ci":     [n_lo, n_hi],
        "residual":     resid_pt,   "residual_ci": [r_lo, r_hi],
        "excludes_zero":        bool(r_lo > 0),
        "null_brackets_chance": bool(n_lo <= 0.50 <= n_hi),
        "intervals_disjoint":   bool(m_lo > n_hi),
        # Task D acceptance-criterion helper: does the point estimate land
        # comfortably inside its own bootstrap interval, or hug an edge?
        # Now checked for all THREE columns, not just the marginals.
        "matched_pt_ok":  bool(m_lo < matched_pt < m_hi),
        "null_pt_ok":     bool(n_lo < null_pt < n_hi),
        "residual_pt_ok": bool(r_lo < resid_pt < r_hi),
        "n_boot": N_BOOT,
        "n_pt_draws": N_PT_DRAWS,
    }


# ---- resume support -------------------------------------------------------
results = {}
if OUTFILE.exists():
    results = json.load(open(OUTFILE))
    print(f"Resuming -- already done: {list(results.keys())}")

In [ ]:
for m in MODELS:
    if m in results:
        print(f"  {m}: already done, skipping")
        continue

    print(f"\n{'='*60}\n{m}\n{'='*60}")
    rev = load_rev(m)
    fil = load_fil(m)
    nl  = rev[0].shape[0]
    print(f"  revisions={len(rev)}  fillers={len(fil)}")

    L, a, profile, near_peak, is_flat = peak_layer(rev, fil, nl)
    print(f"  peak layer = {L}/{nl}  (matched acc={a:.3f})")
    print(f"  layers within {FLAT_WINDOW} of peak: {near_peak}")
    print(f"  {'FLAT PROFILE -- argmax is likely not a real peak' if is_flat else 'looks like a genuine peak'}")

    print(f"  bootstrapping (N_BOOT={N_BOOT}, group-aware CV, "
          f"point estimate averaged over {N_PT_DRAWS} draws)...")
    r = bootstrap_all(rev, fil, L)

    print(f"    matched  = {r['matched']:.3f}  "
          f"[{r['matched_ci'][0]:.3f}, {r['matched_ci'][1]:.3f}]  pt_inside_ci={r['matched_pt_ok']}")
    print(f"    null-fil = {r['null_fil']:.3f}  "
          f"[{r['null_ci'][0]:.3f}, {r['null_ci'][1]:.3f}]"
          f"   brackets 0.50: {r['null_brackets_chance']}  pt_inside_ci={r['null_pt_ok']}")
    print(f"    RESIDUAL = {r['residual']:+.3f}  "
          f"[{r['residual_ci'][0]:+.3f}, {r['residual_ci'][1]:+.3f}]"
          f"   excludes 0: {r['excludes_zero']}")
    print(f"    matched/null intervals disjoint: {r['intervals_disjoint']}")

    results[m] = {"peak_layer": L, "layer_profile": profile,
                  "near_peak_layers": near_peak, "flat_profile": is_flat,
                  "n_rev": len(rev), "n_fil": len(fil), **r}
    json.dump(results, open(OUTFILE, "w"), indent=2)
    del rev, fil
    gc.collect()


# ---- SUMMARY --------------------------------------------------------------
print("\n" + "#"*78)
print(f"#  TASK 1 v3 (N_BOOT={N_BOOT}, group-aware CV, averaged point estimate)")
print("#"*78)
print(f"\n  {'Model':<16} {'matched [95% CI]':>24} {'null [95% CI]':>24} "
      f"{'residual [95% CI]':>26}")
print("  " + "-"*92)
for m in MODELS:
    if m not in results:
        continue
    r = results[m]
    short = m.replace("DeepSeek-R1-Distill-", "R1-")
    print(f"  {short:<16} "
          f"{r['matched']:.3f} [{r['matched_ci'][0]:.3f},{r['matched_ci'][1]:.3f}]".rjust(24)
          + "  " +
          f"{r['null_fil']:.3f} [{r['null_ci'][0]:.3f},{r['null_ci'][1]:.3f}]".rjust(22)
          + "  " +
          f"{r['residual']:+.3f} [{r['residual_ci'][0]:+.3f},{r['residual_ci'][1]:+.3f}]".rjust(24))

print("\n  ACCEPTANCE CRITERIA (Task D)")
print("  " + "-"*76)

In [ ]:
for m in MODELS:
    if m not in results:
        continue
    r = results[m]
    short = m.replace("DeepSeek-R1-Distill-", "R1-")
    print(f"    {short:<16} pt-in-CI(matched/null/residual): "
          f"{str(r['matched_pt_ok']):>5}/{str(r['null_pt_ok']):>5}/{str(r['residual_pt_ok']):>5}   "
          f"residual excl. 0: {str(r['excludes_zero']):>5}   "
          f"null brackets .50: {str(r['null_brackets_chance']):>5}   "
          f"flat peak profile: {str(r['flat_profile']):>5}")

n_pass    = sum(results[m]["excludes_zero"] for m in results)
n_null_ok = sum(results[m]["null_brackets_chance"] for m in results)
n_pt_ok   = sum(results[m]["matched_pt_ok"] and results[m]["null_pt_ok"]
                and results[m]["residual_pt_ok"] for m in results)
print(f"\n  {n_pass}/{len(results)} residual CIs exclude zero.")
print(f"  {n_null_ok}/{len(results)} null CIs bracket chance (0.50) as they should.")
print(f"  {n_pt_ok}/{len(results)} models have BOTH point estimates comfortably inside their CIs.")
if n_pass == len(results) and n_null_ok == len(results) and n_pt_ok == len(results):
    print("  >>> GO. All Task D acceptance criteria met (duplicate-leakage-free).")
else:
    print("  >>> One or more acceptance criteria failed -- see per-model detail above "
          "before writing anything up.")